In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import accuracy_score, r2_score

In [3]:
# Upload your dataset
file_path = "car_price_dataset.csv"   

data = pd.read_csv(file_path)
data.head()

,Unnamed: 0,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,seats,max_power (in bph),Mileage Unit,Mileage,Engine (CC)
0,0,Maruti,2014,450000,145500,Diesel,Individual,Manual,First Owner,5,74.00,kmpl,23.40,1248
1,2,Hyundai,2010,225000,127000,Diesel,Individual,Manual,First Owner,5,90.00,kmpl,23.00,1396
2,4,Hyundai,2017,440000,45000,Petrol,Individual,Manual,First Owner,5,81.86,kmpl,20.14,1197
3,7,Toyota,2011,350000,90000,Diesel,Individual,Manual,First Owner,5,67.10,kmpl,23.59,1364
4,8,Ford,2013,200000,169000,Diesel,Individual,Manual,First Owner,5,68.10,kmpl,20.00,1399


In [4]:
target = input("Enter your target column name: ")

Enter your target column name:  selling_price


In [5]:
print("Dataset Shape:", data.shape)
print("\nData Types:\n", data.dtypes)
print("\nMissing Values:\n", data.isnull().sum())

Dataset Shape: (2095, 14)

Data Types:
 Unnamed: 0              int64
name                   object
year                    int64
selling_price           int64
km_driven               int64
fuel                   object
seller_type            object
transmission           object
owner                  object
seats                   int64
max_power (in bph)    float64
Mileage Unit           object
Mileage               float64
Engine (CC)             int64
dtype: object

Missing Values:
 Unnamed: 0            0
name                  0
year                  0
selling_price         0
km_driven             0
fuel                  0
seller_type           0
transmission          0
owner                 0
seats                 0
max_power (in bph)    0
Mileage Unit          0
Mileage               0
Engine (CC)           0
dtype: int64


In [6]:
# 1. Separate
X = data.drop(columns=[target])
y = data[target]

from sklearn.impute import SimpleImputer
import numpy as np

# 2. Detect column types
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

# 3. Handle numeric columns
if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy='mean')
    X[num_cols] = num_imputer.fit_transform(X[num_cols])

# 4. Handle categorical columns
if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

    # Encode categorical columns
    from sklearn.preprocessing import LabelEncoder
    for col in cat_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

In [7]:
if y.dtype == 'object':
    y = y.astype('category').cat.codes

In [8]:
if len(np.unique(y)) <= 10:
    problem_type = "classification"
else:
    problem_type = "regression"

print("Detected Problem Type:", problem_type)

Detected Problem Type: regression


In [9]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [10]:
if problem_type == "classification":
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "KNN": KNeighborsClassifier(),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "SVM": SVC()
    }

else:
    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(),
        "Random Forest": RandomForestRegressor()
    }

In [11]:
results = {}

for name, model in models.items():
    try:
        scores = cross_val_score(model, X_scaled, y, cv=5)
        results[name] = scores.mean()
    except Exception as e:
        results[name] = 0

In [12]:
results_df = pd.DataFrame(list(results.items()), columns=["Algorithm", "Score"])
results_df = results_df.sort_values(by="Score", ascending=False)

results_df

,Algorithm,Score
2,Random Forest,0.884645
1,Decision Tree,0.788115
0,Linear Regression,0.739185


In [13]:
best_algorithm = results_df.iloc[0]

print("🏆 Best Algorithm:", best_algorithm["Algorithm"])
print("📊 Score:", best_algorithm["Score"])

🏆 Best Algorithm: Random Forest
📊 Score: 0.8846452394310657
